In [59]:
# Paths and metadata
from pathlib import Path
import json, re
import pandas as pd

wvs_dir = Path("WorldValuesBench-main")
raw_fn = wvs_dir / "WVS_Cross-National_Wave_7_csv_v6_0.csv"
country, country_code = "US", 840
long_fn = Path(f"wvs7_{country}_long.parquet")

meta = json.load(open(wvs_dir / "WorldValuesBench/question_metadata.json"))          # question text; which columns are value questions
codebook = json.load(open(wvs_dir / "dataset_construction/codebook.json"))           # code -> label for every column
adjusted = json.load(open(wvs_dir / "dataset_construction/answer_adjustment.json"))  # WorldValuesBench recoding: old code -> new code ("" = dropped)
questions = [k for k, m in meta.items() if m["use_case"] == "value" and k not in ("Q171", "Q97", "Q240", "Q222")]   # Q171 (religious attendance), Q97 (labor union membership), Q240 (left-right), Q222 (votes in national elections): persona columns instead
len(questions)

235

In [60]:
# US answers and demographics from the raw file (original codes; negative codes = missing)
demographics = [k for k, m in meta.items() if m["use_case"] == "demographic" and k not in ("D_INTERVIEW", "E1_LITERACY")] + ["N_REGION_ISO", "Q223", "Q171", "Q97", "Q240", "Q222"]   # E1_LITERACY: not asked in the US; + region, party vote, religious attendance, union membership, left-right, national voting
raw = pd.read_csv(raw_fn, usecols=["D_INTERVIEW", *demographics, *questions])
raw = raw[raw["B_COUNTRY"] == country_code].reset_index(drop=True)
print(f"{country} respondents: {len(raw)}, demographic columns: {len(demographics)}, value questions: {len(questions)}")

US respondents: 2596, demographic columns: 47, value questions: 235


In [61]:
# WorldValuesBench recoding for the questions it adjusted: options reordered, merged or dropped; a dropped answer -> missing
def offered(v):
    """{code: label} of every option after WorldValuesBench's recoding, in code order; on a merge the option that keeps its code keeps its label"""
    adj, opts = adjusted.get(v, {}), {}
    for c in sorted(codebook[v]["choices"], key=int):
        if int(c) < 0 or adj.get(c, c) == "":
            continue
        new = int(adj.get(c, c))
        if new not in opts or str(new) == c:
            opts[new] = codebook[v]["choices"][c]
    return dict(sorted(opts.items()))

answers = raw[["D_INTERVIEW", *questions]].copy()
for v in [q for q in adjusted if q in questions]:
    mapping = {int(o): (int(n) if n != "" else -9) for o, n in adjusted[v].items()}   # -9: dropped answer -> missing
    answers[v] = answers[v].map(lambda x: mapping.get(x, x))

In [62]:
# Before -> after for the questions WorldValuesBench recoded
for v in [q for q in adjusted if q in questions]:
    after = offered(v)
    print("=" * 110)
    print(f"{v} | {codebook[v]['question']}")
    print(f"question: {meta[v]['question']}")
    print("-" * 110)
    for c, t in sorted(codebook[v]["choices"].items(), key=lambda x: int(x[0])):
        if int(c) < 0:
            continue
        new = adjusted[v].get(c, c)
        print(f"{c:>4}  {t:<50} ->  " + (f"{new:>2}  {after[int(new)]}" if new != "" else "dropped (answer -> missing)"))
print("=" * 110)

Q56 | Standard of living comparing with your parents
question: On a scale of 1 to 3, 1 meaning 'Better off' and 3 meaning 'Worse Off', how would you rate your current standard of living compared to your parents standard of living when they were your age?
--------------------------------------------------------------------------------------------------------------
   1  Better off                                         ->   1  Better off
   2  Worse off                                          ->   3  Worse off
   3  Or about the same                                  ->   2  Or about the same
Q111 | Protecting environment vs. Economic growth
question: On a scale of 1 to 2, 1 meaning 'Protecting the environment should be given priority, even if it causes slower economic growth and some loss of jobs' and 2 meaning 'Economic growth and creating jobs should be the top priority, even if the environment suffers to some extent.', where does your opinion fall?
---------------------------------

In [63]:
# Question bank: WorldValuesBench text; options after WorldValuesBench's recoding, in code order
bank = pd.DataFrame({
    "v": questions,
    "label": [re.sub(r"\s+", " ", codebook[v]["question"]).strip() for v in questions],
    "question": [meta[v]["question"] for v in questions],
    "type": ["choice" if v == "Q119" else "score" for v in questions],   # Q119: "Hard to say" is not a scale point
    "reversed": pd.NA,
    "option_codes": [list(offered(v)) for v in questions],
    "option_texts": [list(offered(v).values()) for v in questions],
})
bank

,v,label,question,type,reversed,option_codes,option_texts
0,Q1,Important in life: Family,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im..."
1,Q2,Important in life: Friends,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im..."
2,Q3,Important in life: Leisure time,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im..."
3,Q4,Important in life: Politics,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im..."
4,Q5,Important in life: Work,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im..."
...,...,...,...,...,...,...,...
230,Q255,"Feel close to your village, town or city","On a scale of 1 to 4, 1 meaning 'Very close' a...",score,<NA>,"[1, 2, 3, 4]","[Very close, Close, Not very close, Not close ..."
231,Q256,"Feel close to your district, region","On a scale of 1 to 4, 1 meaning 'Very close' a...",score,<NA>,"[1, 2, 3, 4]","[Very close, Close, Not very close, Not close ..."
232,Q257,Feel close to your country,"On a scale of 1 to 4, 1 meaning 'Very close' a...",score,<NA>,"[1, 2, 3, 4]","[Very close, Close, Not very close, Not close ..."
233,Q258,Feel close to your continent,"On a scale of 1 to 4, 1 meaning 'Very close' a...",score,<NA>,"[1, 2, 3, 4]","[Very close, Close, Not very close, Not close ..."


In [64]:
# Demographics under their WVS names (mapped to global personas in persona.py): code -> codebook label; a column without labels keeps its numbers; negative codes -> NaN
persona = raw[["D_INTERVIEW", *demographics]].rename(columns={"D_INTERVIEW": "CASEID"})
for c in demographics:
    labels = {k: t for k, t in codebook[c]["choices"].items() if int(k) >= 0}
    if labels:
        persona[c] = [labels.get(str(int(x)), str(int(x))) if x >= 0 else float("nan") for x in persona[c]]
    else:
        persona[c] = persona[c].where(persona[c] >= 0)
persona

,CASEID,A_YEAR,B_COUNTRY,J_INTDATE,N_TOWN,G_TOWNSIZE,H_SETTLEMENT,H_URBRURAL,S_INTLANGUAGE,Q260,...,Q288,Q289,Q289CS9,Q290,N_REGION_ISO,Q223,Q171,Q97,Q240,Q222
0,840071001,2017,United States,20170515,840004,"100,000-500,000",District center,Urban,English,Female,...,Third step,NaN,NaN,"US: Other, Non-Hispanic",US-AL Alabama,USA: Democratic Party,Once a week,Not a member,4,Never
1,840071002,2017,United States,20170501,840178,"20,000-50,000","Another city, town (not a regional or district...",Urban,English,Female,...,Fifth step,Do not belong to a denomination,Non-religious,"US: White, non-Hispanic",US-LA Louisiana,Other,Once a week,Not a member,5,Never
2,840071003,2017,United States,20170501,840137,"20,000-50,000","Another city, town (not a regional or district...",Urban,English,Female,...,Lower step,Do not belong to a denomination,Non-religious,"US: White, non-Hispanic",US-GA Georgia,NaN,Only on special holy days,Not a member,Right,Never
3,840071004,2017,United States,20170502,840214,"100,000-500,000",District center,Urban,English,Female,...,Lower step,Do not belong to a denomination,Non-religious,"US: White, non-Hispanic",US-MO Missouri,Other,"Never, practically never",Not a member,5,Never
4,840071005,2017,United States,20170502,840224,"50,000-100,000","Another city, town (not a regional or district...",Urban,English,Female,...,Fifth step,Do not belong to a denomination,Non-religious,"US: White, non-Hispanic",US-NC North Carolina,USA: Republican Party,Only on special holy days,Not a member,Left,Never
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2591,840073592,2017,United States,20170502,840022,"50,000-100,000","Another city, town (not a regional or district...",Urban,English,Female,...,Fourth step,Roman Catholic,Roman Catholic; Latin Church;,US: Hispanic,US-CA California,USA: Democratic Party,"Never, practically never",Not a member,Left,Never
2592,840073593,2017,United States,20170501,840264,"500,000 and more",District center,Urban,English,Male,...,Fourth step,NaN,NaN,US: Hispanic,US-NY New York,Other,"Never, practically never",Inactive member,5,Never
2593,840073594,2017,United States,20170509,840194,NaN,"Another city, town (not a regional or district...",Urban,English,Female,...,Fifth step,Do not belong to a denomination,Non-religious,"US: White, non-Hispanic",US-ME Maine,USA: Democratic Party,"Never, practically never",Not a member,NaN,Always
2594,840073595,2017,United States,20170509,840023,"100,000-500,000",District center,Urban,English,Male,...,Eight step,Do not belong to a denomination,Non-religious,"US: Other, Non-Hispanic",US-CA California,USA: Democratic Party,"Never, practically never",NaN,3,Usually


In [65]:
persona.columns

Index(['CASEID', 'A_YEAR', 'B_COUNTRY', 'J_INTDATE', 'N_TOWN', 'G_TOWNSIZE',
       'H_SETTLEMENT', 'H_URBRURAL', 'S_INTLANGUAGE', 'Q260', 'Q261', 'Q262',
       'X003R', 'Q263', 'Q264', 'Q265', 'Q266', 'Q267', 'Q268', 'Q269', 'Q270',
       'Q271', 'Q272', 'Q273', 'Q274', 'Q275', 'Q276', 'Q277', 'Q278', 'Q279',
       'Q280', 'Q281', 'Q282', 'Q283', 'Q284', 'Q285', 'Q286', 'Q287', 'Q288',
       'Q289', 'Q289CS9', 'Q290', 'N_REGION_ISO', 'Q223', 'Q171', 'Q97',
       'Q240', 'Q222'],
      dtype='str')

In [66]:
for c in persona.columns:
    print(c)
    print(persona[c].value_counts())
    print('+'*50)


CASEID
CASEID
840071001    1
840071002    1
840071003    1
840071004    1
840071005    1
            ..
840073592    1
840073593    1
840073594    1
840073595    1
840073596    1
Name: count, Length: 2596, dtype: int64
++++++++++++++++++++++++++++++++++++++++++++++++++
A_YEAR
A_YEAR
2017    2596
Name: count, dtype: int64
++++++++++++++++++++++++++++++++++++++++++++++++++
B_COUNTRY
B_COUNTRY
United States    2596
Name: count, dtype: int64
++++++++++++++++++++++++++++++++++++++++++++++++++
J_INTDATE
J_INTDATE
20170501    933
20170502    473
20170503    277
20170504    162
20170428    110
20170505     66
20170510     60
20170509     54
20170515     50
20170429     45
20170511     35
20170506     35
20170507     30
20170523     23
20170519     22
20170508     22
20170517     21
20170516     20
20170522     19
20170512     17
20170513     17
20170430     17
20170518     15
20170514     14
20170524      9
20170521      9
20170520      9
20170531      8
20170525      6
20170528      5
2017052

In [67]:
# Long table: one row per answer, with persona, question and every option offered
long = answers.rename(columns={"D_INTERVIEW": "CASEID"}).melt(id_vars="CASEID", var_name="v", value_name="answer_code")
long = long[long["answer_code"] >= 0]
long = long.merge(persona, on="CASEID").merge(bank, on="v")
long["answer_text"] = [dict(zip(c, t))[a] for c, t, a in zip(long["option_codes"], long["option_texts"], long["answer_code"])]
long["answer_code"] = long["answer_code"].astype("Int64")
long.head(2)

,CASEID,v,answer_code,A_YEAR,B_COUNTRY,J_INTDATE,N_TOWN,G_TOWNSIZE,H_SETTLEMENT,H_URBRURAL,...,Q97,Q240,Q222,label,question,type,reversed,option_codes,option_texts,answer_text
0,840071001,Q1,1,2017,United States,20170515,840004,"100,000-500,000",District center,Urban,...,Not a member,4,Never,Important in life: Family,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im...",Very important
1,840071002,Q1,1,2017,United States,20170501,840178,"20,000-50,000","Another city, town (not a regional or district...",Urban,...,Not a member,5,Never,Important in life: Family,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im...",Very important


In [68]:
# Export for training
columns = ["CASEID", *demographics, "v", "label", "question", "type", "reversed",
           "option_codes", "option_texts", "answer_code", "answer_text"]
long[columns].to_parquet(long_fn, index=False)
long_fn

PosixPath('wvs7_US_long.parquet')

In [69]:
long[columns]

,CASEID,A_YEAR,B_COUNTRY,J_INTDATE,N_TOWN,G_TOWNSIZE,H_SETTLEMENT,H_URBRURAL,S_INTLANGUAGE,Q260,...,Q222,v,label,question,type,reversed,option_codes,option_texts,answer_code,answer_text
0,840071001,2017,United States,20170515,840004,"100,000-500,000",District center,Urban,English,Female,...,Never,Q1,Important in life: Family,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im...",1,Very important
1,840071002,2017,United States,20170501,840178,"20,000-50,000","Another city, town (not a regional or district...",Urban,English,Female,...,Never,Q1,Important in life: Family,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im...",1,Very important
2,840071003,2017,United States,20170501,840137,"20,000-50,000","Another city, town (not a regional or district...",Urban,English,Female,...,Never,Q1,Important in life: Family,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im...",1,Very important
3,840071004,2017,United States,20170502,840214,"100,000-500,000",District center,Urban,English,Female,...,Never,Q1,Important in life: Family,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im...",1,Very important
4,840071005,2017,United States,20170502,840224,"50,000-100,000","Another city, town (not a regional or district...",Urban,English,Female,...,Never,Q1,Important in life: Family,"On a scale of 1 to 4, 1 meaning 'Very importan...",score,<NA>,"[1, 2, 3, 4]","[Very important, Rather important, Not very im...",1,Very important
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595278,840073592,2017,United States,20170502,840022,"50,000-100,000","Another city, town (not a regional or district...",Urban,English,Female,...,Never,Q259,Feel close to the world,"On a scale of 1 to 4, 1 meaning 'Very close' a...",score,<NA>,"[1, 2, 3, 4]","[Very close, Close, Not very close, Not close ...",2,Close
595279,840073593,2017,United States,20170501,840264,"500,000 and more",District center,Urban,English,Male,...,Never,Q259,Feel close to the world,"On a scale of 1 to 4, 1 meaning 'Very close' a...",score,<NA>,"[1, 2, 3, 4]","[Very close, Close, Not very close, Not close ...",2,Close
595280,840073594,2017,United States,20170509,840194,NaN,"Another city, town (not a regional or district...",Urban,English,Female,...,Always,Q259,Feel close to the world,"On a scale of 1 to 4, 1 meaning 'Very close' a...",score,<NA>,"[1, 2, 3, 4]","[Very close, Close, Not very close, Not close ...",2,Close
595281,840073595,2017,United States,20170509,840023,"100,000-500,000",District center,Urban,English,Male,...,Usually,Q259,Feel close to the world,"On a scale of 1 to 4, 1 meaning 'Very close' a...",score,<NA>,"[1, 2, 3, 4]","[Very close, Close, Not very close, Not close ...",4,Not close at all
